# Recreate an image with a mosaic of images

## Notes:
- Here, this script will work better with images with same aspect ratio
- ToDo: make a script that can create a grid with images from different aspect ratio

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
# from pathlib import Path
from datetime import date
# from dataclasses import dataclass, field
# import itertools


# import 3rd-party modules
import cv2
import numpy as np
from numba import njit
import ray
# from pygifsicle import optimize


# import local modules
# from core.utils.renderer.giffer import create_gif
# from core.utils.renderer.videographer import create_video
from core.utils.project_manager import Project
from core.utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [2]:
# name out img dirs
out_img_dir_list = ["mosaic", "mosaic_with_duplicate", "mosaic_blend"]

# create project
project = Project(project_dir="assets/images/mosaic/african_textile", in_img_dir="db", out_img_dir_list=out_img_dir_list)

## Define functions and classes

In [3]:
# decorate function with numba fct to speed up execution
@njit()
def create_mosaic(src_imgs, dest_img, cell_shape, nb_cols, nb_rows):
    """
    Function to find best matching image for region of interest of another image
    Important: The images should be in LAB color space for better results.

    Arguments
    * src_img: source image
    * dest_img: destination image, i.e. image to recreate with images
    """
    # unpack grid cell image shape
    cell_height, cell_width = cell_shape[:2]
    
    # create empty lists (or arrays) to store coords mapping between source and dest images
    src_imgs_idxs_grid_yxs = []

    # get list of grid positions
    grid_positions = [(grid_y, grid_x) for grid_y in range(nb_cols) for grid_x in range(nb_rows)]

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # choose random seed to recreate same shuffle or change it to see if you get better results
    np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    np.random.shuffle(grid_positions_idxs)

    # iterate over each grid position
    for i in grid_positions_idxs:

        grid_y, grid_x = grid_positions[i]
        
        # get region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi = dest_img[y:y+cell_height, x:x+cell_width]

        # inititiate trackers for best match to this roi
        best_match_dist = np.inf
        best_match_index = 0

        # iterate over each image in list of source images
        for src_img_idx, src_img in enumerate(src_imgs):

            # compute distance between the pixels
            dist = np.sum(np.abs(dest_roi - src_img))
            
            # if distance is smaller than the current best dist
            if dist < best_match_dist:
                # update current best dist
                best_match_dist = dist
                # store best match index 
                best_match_index = src_img_idx

        # take out best match image from list of source images
        best_match_img = src_imgs.pop(best_match_index)

        # append best match index and its corresponding grid position to list
        src_imgs_idxs_grid_yxs.append((best_match_index, grid_y, grid_x))

        # update roi with best match image
        dest_img[y:y + ref_img_height, x:x + ref_img_width] = best_match_img

    return dest_img, src_imgs_idxs_grid_yxs

In [18]:
# decorate function with numba fct to speed up execution
# @njit()
def create_mosaic_duplicate_img(src_imgs, dest_img, cell_shape, nb_cols, nb_rows, src_imgs_subset=None):
    """
    Function to find best matching image for region of interest of another image
    Important: The images should be in LAB color space for better results.

    Arguments
    * src_img: source image
    * dest_img: destination image, i.e. image to recreate with images
    """
    # unpack grid cell image shape
    cell_height, cell_width = cell_shape[:2]
    
    # create empty lists (or arrays) to store coords mapping between source and dest images
    src_imgs_idxs_grid_yxs = []

    # get list of grid positions
    grid_positions = [(grid_y, grid_x) for grid_y in range(nb_cols) for grid_x in range(nb_rows)]

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # # choose random seed to recreate same shuffle or change it to see if you get better results
    # np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    np.random.shuffle(grid_positions_idxs)

    # get index range of grid positions
    src_imgs_idxs = np.arange(len(src_imgs))

    if src_imgs_subset is None:
        src_imgs_subset = len(src_imgs_idxs)

    # iterate over each grid position
    for i in grid_positions_idxs:

        # shuffle grid positions (otherwise the first grids from top will get the best matching images)
        np.random.shuffle(src_imgs_idxs)

        grid_y, grid_x = grid_positions[i]
        
        # get region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi = dest_img[y:y+cell_height, x:x+cell_width]

        # inititiate trackers for best match to this roi
        best_match_dist = np.inf
        best_match_index = 0

        # iterate over each image in list of source images
        for src_img_idx in src_imgs_idxs[:src_imgs_subset]:

            # get src img
            src_img = src_imgs[src_img_idx]

            # loop through degrees
            for degree_90_factor in range(4):

                # rotate image by x times 90°
                src_img = np.rot90(src_img, degree_90_factor)

                # compute distance between the pixels
                dist = np.sum(np.abs(dest_roi - src_img))
                
                # if distance is smaller than the current best dist
                if dist < best_match_dist:
                    # update current best dist
                    best_match_dist = dist
                    # store best match index & degree
                    best_match_index = src_img_idx
                    best_degree = degree_90_factor

        # # take out best match image from list of source images
        # best_match_img = src_imgs.pop(best_match_index)
        best_match_img = src_imgs[best_match_index]
        
        # rotate image by x times 90°
        best_match_img = np.rot90(best_match_img, best_degree)

        # append best match index and its corresponding grid position to list
        src_imgs_idxs_grid_yxs.append((best_match_index, grid_y, grid_x))

        # update roi with best match image
        dest_img[y:y + ref_img_height, x:x + ref_img_width] = best_match_img

    return dest_img

## Define variables & constants

In [9]:
# set grid caracteristics
NB_ROWS = 120
NB_COLS = 120

# set dest image path (i.e path of image to recreate)
dest_img_path = "assets/images/mosaic/african_textile/mman-2022.JPG"

## Read images

In [10]:
# read first image to get a reference shape
ref_img = cv2.imread(project.in_img_path_list[0])
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape
# set new ref height & width to save memory if necessary
ref_img_height, ref_img_width = ref_img_height//1, ref_img_width//1

# set output grid image shape
out_img_height = ref_img_height * NB_ROWS
out_img_width = ref_img_width * NB_COLS
out_img_channel = ref_img_channel

# read images to use to recreate dest image
src_imgs = [cv2.resize(cv2.cvtColor(cv2.imread(src_img_path), cv2.COLOR_BGR2LAB), (ref_img_width, ref_img_height)) for src_img_path in project.in_img_path_list]

# get dest image and resize to output grid image shape
dest_img = cv2.resize(cv2.cvtColor(cv2.imread(dest_img_path), cv2.COLOR_BGR2LAB), (out_img_width, out_img_height))

In [31]:
# set reference shape
ref_img_height, ref_img_width, ref_img_channel = 50, 50, 3

# set output grid image shape
out_img_height = ref_img_height * NB_ROWS
out_img_width = ref_img_width * NB_COLS
out_img_channel = ref_img_channel



# read images to use to recreate dest image
src_imgs = [resize_with_crop(img=cv2.cvtColor(cv2.imread(src_img_path), cv2.COLOR_BGR2LAB), ref_img_shape=(ref_img_width, ref_img_height, ref_img_channel)) for src_img_path in project.in_img_path_list[:5]]

# get dest image and resize to output grid image shape
dest_img = resize_with_crop(img=cv2.cvtColor(cv2.imread(dest_img_path), cv2.COLOR_BGR2LAB), ref_img_shape=(out_img_width, out_img_height, out_img_channel))

dest_img = cv2.bilateralFilter(dest_img,9,150,150)

img: (965, 633), ref_img: (50, 50)
img_ratio: 0.655958549222798, ref_img_ratio: 1.0
img: (636, 1020), ref_img: (50, 50)
img_ratio: 1.6037735849056605, ref_img_ratio: 1.0
img: (565, 945), ref_img: (50, 50)
img_ratio: 1.6725663716814159, ref_img_ratio: 1.0
img: (1290, 695), ref_img: (50, 50)
img_ratio: 0.5387596899224806, ref_img_ratio: 1.0
img: (1333, 826), ref_img: (50, 50)
img_ratio: 0.6196549137284321, ref_img_ratio: 1.0
img: (1767, 1325), ref_img: (6000, 6000)
img_ratio: 0.7498585172608941, ref_img_ratio: 1.0


## Recreate image

In [16]:
out_img, src_imgs_idxs_grid_yxs = create_mosaic(src_imgs, dest_img, cell_shape=(ref_img_height, ref_img_width), nb_cols=NB_COLS, nb_rows=NB_ROWS)

# convert image to bgr
out_img = cv2.cvtColor(out_img, cv2.COLOR_LAB2BGR)

# set output image directory
out_img_dir = "mosaic"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{NB_ROWS*NB_ROWS}_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

/Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'src_imgs' of function 'create_mosaic'.

For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types

File "<ipython-input-3-0f86a17b20be>", line 3:
@njit()
def create_mosaic(src_imgs, dest_img, cell_shape, nb_cols, nb_rows):
^

  warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))


IndexError: pop from empty list

In [19]:
# src_imgs_subset = len(src_imgs)//10

# out_img, src_imgs_idxs_grid_yxs = create_mosaic_duplicate_img(src_imgs, dest_img, cell_shape=(ref_img_height, ref_img_width), nb_cols=NB_COLS, nb_rows=NB_ROWS, src_imgs_subset=src_imgs_subset)
out_img = create_mosaic_duplicate_img(src_imgs, dest_img, cell_shape=(ref_img_height, ref_img_width), nb_cols=NB_COLS, nb_rows=NB_ROWS)

# convert image to bgr
out_img = cv2.cvtColor(out_img, cv2.COLOR_LAB2BGR)

# set output image directory
out_img_dir = "mosaic_with_duplicate"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{NB_ROWS*NB_ROWS}_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

True

### with multiprocessing

In [24]:
# decorate function with numba fct to speed up execution
# @njit()
def create_mosaic_duplicate_img(src_imgs, dest_img, cell_shape, nb_rows, nb_cols, col_range=None, src_imgs_subset=None):
    """
    Function to find best matching image for region of interest of another image
    Important: The images should be in LAB color space for better results.

    Arguments
    * src_img: source image
    * dest_img: destination image, i.e. image to recreate with images
    """
    # dest_img = dest_img.copy()
    canvas = np.zeros_like(dest_img)

    # unpack grid cell image shape
    cell_height, cell_width = cell_shape[:2]

    # get list of grid positions
    if col_range is not None:
        grid_positions = [(grid_y, grid_x) for grid_y in range(*col_range) for grid_x in range(nb_rows)]
    else:
        grid_positions = [(grid_y, grid_x) for grid_y in range(nb_cols) for grid_x in range(nb_rows)]

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # # choose random seed to recreate same shuffle or change it to see if you get better results
    # np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    np.random.shuffle(grid_positions_idxs)

    # get index range of grid positions
    src_imgs_idxs = np.arange(len(src_imgs))

    if src_imgs_subset is None:
        src_imgs_subset = len(src_imgs_idxs)

    # iterate over each grid position
    for i in grid_positions_idxs:

        # shuffle grid positions (otherwise the first grids from top will get the best matching images)
        np.random.shuffle(src_imgs_idxs)

        grid_y, grid_x = grid_positions[i]
        
        # get region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi = dest_img[y:y+cell_height, x:x+cell_width]

        # inititiate trackers for best match to this roi
        best_match_dist = np.inf
        best_match_index = 0

        # iterate over each image in list of source images
        for src_img_idx in src_imgs_idxs[:src_imgs_subset]:

            # get src img
            src_img = src_imgs[src_img_idx]

            # loop through degrees
            for degree_90_factor in range(4):

                # rotate image by x times 90°
                src_img = np.rot90(src_img, degree_90_factor)

                # compute distance between the pixels
                dist = np.sum(np.abs(dest_roi - src_img))
                
                # if distance is smaller than the current best dist
                if dist < best_match_dist:
                    # update current best dist
                    best_match_dist = dist
                    # store best match index & degree
                    best_match_index = src_img_idx
                    best_degree = degree_90_factor

        # # take out best match image from list of source images
        # best_match_img = src_imgs.pop(best_match_index)
        best_match_img = src_imgs[best_match_index]
        
        # rotate image by x times 90°
        best_match_img = np.rot90(best_match_img, best_degree)

        # update roi with best match image
        # dest_img[y:y + ref_img_height, x:x + ref_img_width] = best_match_img
        canvas[y:y + ref_img_height, x:x + ref_img_width] = best_match_img

    return canvas

In [32]:
# shutdown ray
ray.shutdown()

# start ray
ray.init(log_to_driver=True)

@ray.remote
def multiprocess(fct, **kwargs):
    return fct(**kwargs)

In [33]:
# choose number of tasks
NB_TASKS = 4

# src_imgs_subset = len(src_imgs)//100

# start tasks in parallel
result_ids = []

for worker_nb, i in enumerate(range(0, NB_COLS, NB_COLS//NB_TASKS)): # set grid range for each worker

    # recreate given image with images mosaic
    # result_ids.append(multiprocess.remote(create_mosaic_duplicate_img, src_imgs=src_imgs, dest_img=dest_img, cell_shape=(ref_img_height, ref_img_width), nb_rows=NB_ROWS, nb_cols=NB_COLS, col_range=(i, i+NB_COLS//NB_TASKS), src_imgs_subset=src_imgs_subset))
    # result_ids.append(multiprocess.remote(create_mosaic_duplicate_img, src_imgs=src_imgs, dest_img=dest_img[:, worker_nb*out_img_width//NB_TASKS:(worker_nb+1)*out_img_width//NB_TASKS], cell_shape=(ref_img_height, ref_img_width), nb_rows=NB_ROWS, nb_cols=NB_COLS, col_range=(i, i+NB_COLS//NB_TASKS)))
    result_ids.append(multiprocess.remote(create_mosaic_duplicate_img, src_imgs=src_imgs, dest_img=dest_img, cell_shape=(ref_img_height, ref_img_width), nb_rows=NB_ROWS, nb_cols=NB_COLS, col_range=(i, i+NB_COLS//NB_TASKS)))

# wait for the tasks to complete and retrieve the results
results = ray.get(result_ids)

out_img = np.zeros_like(dest_img)

for result in results:
    out_img = cv2.add(out_img, result)

# out_img = cv2.hconcat(results)

# convert image to bgr
out_img = cv2.cvtColor(out_img, cv2.COLOR_LAB2BGR)

# set output image directory
out_img_dir = "mosaic_with_duplicate"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{NB_ROWS*NB_ROWS}_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

# shutdown ray
ray.shutdown()

In [41]:
@njit()
def pop_list(l, i):
    e = l.pop(i)
    return l
    
l = np.arange(12).tolist()
results = ray.get([multiprocess.remote(pop_list, l=l, i=i) for i in range(4)])
results

(multiprocess pid=40992) /Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
(multiprocess pid=40992) Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'l' of function 'pop_list'.
(multiprocess pid=40992) 
(multiprocess pid=40992) For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types
(multiprocess pid=40992) 
(multiprocess pid=40992) File "<ipython-input-38-218e6e97813c>", line 1:
(multiprocess pid=40992) <source missing, REPL/exec in use?>
(multiprocess pid=40992) 
(multiprocess pid=40992)   warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))
(multiprocess pid=40991) /Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
(multiprocess pid=40991) Encountered the use of a type t

[(0, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]),
 (1, [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]),
 (2, [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11]),
 (3, [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11])]

In [53]:
# shutdown ray
ray.shutdown()

# start ray
ray.init(log_to_driver=True)

# decorate class to make instances of this class actors
# @ray.remote(num_cpus=2, num_gpus=0.5) # specify resources for an actor 
@ray.remote
class Mosaic(object):
    def __init__(self, fct, l):
        self.fct = fct
        self.l = l

    def act(self, i):
        self.l = self.fct(l=self.l, i=i)
        return self.l

# choose number of tasks
NB_TASKS = 4
l = np.arange(4).tolist()

# create actors from class
mosaics = [Mosaic.remote(pop_list, l) for _ in range(NB_TASKS)]

# wait for the tasks to complete and retrieve the results
# results = ray.get([mosaic.act.remote(i=i) for i, mosaic in enumerate(mosaics)]) # tasks in parallel
results = ray.get([mosaics[0].act.remote(i=0) for _ in range(NB_TASKS)]) # successive tasks with share state
results

[[1, 2, 3], [2, 3], [3], []]

(Mosaic pid=48267) /Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
(Mosaic pid=48267) Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'l' of function 'pop_list'.
(Mosaic pid=48267) 
(Mosaic pid=48267) For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types
(Mosaic pid=48267) 
(Mosaic pid=48267) File "<ipython-input-45-46e8bc3f4e8b>", line 1:
(Mosaic pid=48267) <source missing, REPL/exec in use?>
(Mosaic pid=48267) 
(Mosaic pid=48267)   warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))


In [10]:
# shutdown ray
ray.shutdown()

# start ray
ray.init(log_to_driver=True)

# decorate class to make instances of this class actors
# @ray.remote(num_cpus=2, num_gpus=0.5) # specify resources for an actor 
@ray.remote
class Mosaic(object):
    def __init__(self, src_imgs, dest_img):
        self.src_imgs = src_imgs
        self.dest_img = dest_img

    def act(self, fct, **kwargs):
        self.dest_img = fct(self.src_imgs, self.dest_img, **kwargs)
        return self.dest_img

In [16]:
for i in range(0, NB_COLS, NB_COLS//NB_TASKS): # set grid range for each worker
    print(NB_ROWS, i,i+NB_COLS//NB_TASKS)

60 0 15
60 15 30
60 30 45
60 45 60


In [14]:
# choose number of tasks
NB_TASKS = 4

src_imgs_subset = len(src_imgs)//100

# start tasks in parallel
mosaics = []
result_ids = []

for i in range(0, NB_COLS, NB_COLS//NB_TASKS): # set grid range for each worker

    # create actor from class
    mosaic = Mosaic.remote(src_imgs, dest_img[:NB_ROWS, i:i+NB_COLS//NB_TASKS])
    # recreate given image with images mosaic
    result_ids.append(mosaic.act.remote(create_mosaic_duplicate_img, cell_shape=(ref_img_height, ref_img_width), nb_rows=NB_ROWS, nb_cols=NB_COLS, col_range=(i, i+NB_COLS//NB_TASKS), src_imgs_subset=src_imgs_subset))

# wait for the tasks to complete and retrieve the results
results = ray.get(result_ids)

out_img = cv2.hconcat(results)

# convert image to bgr
out_img = cv2.cvtColor(out_img, cv2.COLOR_LAB2BGR)

# set output image directory
out_img_dir = "mosaic_with_duplicate"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{NB_ROWS*NB_ROWS}_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

(Mosaic pid=66328) /Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
(Mosaic pid=66328) Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'src_imgs' of function 'create_mosaic_duplicate_img'.
(Mosaic pid=66328) 
(Mosaic pid=66328) For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types
(Mosaic pid=66328) 
(Mosaic pid=66328) File "<ipython-input-6-11eebadfb143>", line 2:
(Mosaic pid=66328) <source missing, REPL/exec in use?>
(Mosaic pid=66328) 
(Mosaic pid=66328)   warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))
2022-03-21 00:40:32,029	ERROR worker.py:84 -- Unhandled error (suppress with RAY_IGNORE_UNHANDLED_ERRORS=1): ray::Mosaic.act() (pid=66328, ip=127.0.0.1, repr=<__main__.Mosaic object at 0x1093141f0>)
  File "<ipython-input-10-7c76d871e2ee>", li

RayTaskError(ValueError): [36mray::Mosaic.act()[39m (pid=66328, ip=127.0.0.1, repr=<__main__.Mosaic object at 0x1093141f0>)
  File "<ipython-input-10-7c76d871e2ee>", line 16, in act
ValueError: unable to broadcast argument 1 to output array
File "/Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/np/npyimpl.py", line 228,

(Mosaic pid=66564) /Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
(Mosaic pid=66564) Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'src_imgs' of function 'create_mosaic_duplicate_img'.
(Mosaic pid=66564) 
(Mosaic pid=66564) For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types
(Mosaic pid=66564) 
(Mosaic pid=66564) File "<ipython-input-6-11eebadfb143>", line 2:
(Mosaic pid=66564) <source missing, REPL/exec in use?>
(Mosaic pid=66564) 
(Mosaic pid=66564)   warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))
2022-03-21 00:41:13,212	ERROR worker.py:84 -- Unhandled error (suppress with RAY_IGNORE_UNHANDLED_ERRORS=1): ray::Mosaic.act() (pid=66564, ip=127.0.0.1, repr=<__main__.Mosaic object at 0x113eac1f0>)
  File "<ipython-input-10-7c76d871e2ee>", li

In [7]:
src_imgs_subset = len(src_imgs)//10

out_img = create_mosaic_duplicate_img(src_imgs, dest_img, cell_shape=(ref_img_height, ref_img_width), nb_rows=NB_ROWS, nb_cols=NB_COLS, col_range=(0,15), src_imgs_subset=src_imgs_subset)

# convert image to bgr
out_img = cv2.cvtColor(out_img, cv2.COLOR_LAB2BGR)

# set output image directory
out_img_dir = "mosaic_with_duplicate"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"test_mosaic_{NB_ROWS*NB_ROWS}_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

/Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'src_imgs' of function 'create_mosaic_duplicate_img'.

For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types

File "<ipython-input-6-11eebadfb143>", line 3:
@njit()
def create_mosaic_duplicate_img(src_imgs, dest_img, cell_shape, nb_rows, nb_cols, col_range=None, src_imgs_subset=None):
^

  warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))


## Blend mosaics with original image

In [2]:
from core.utils.renderer.resizer import resize_with_pad

In [5]:
out_img_dir = "mosaic_blend"
NB_IMGS = 30

src_img = cv2.imread("assets/images/mosaic/snapshots/spiderman/to_recreate/spiderman_road.png")

mosaic_img = resize_with_pad(img_path="/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/mosaic/snapshots/spiderman/mosaic_results_spiderman/matrix_photo_250000_20220210.jpg",
ref_img=src_img)

# get range to set images blending alpha at each step
alpha_range = np.linspace(0, 1, NB_IMGS)

# blend image at each step
for i in range(NB_IMGS):

    # blend image
    alpha = alpha_range[i]
    beta = 1 - alpha
    out_img = cv2.addWeighted(src_img,alpha, mosaic_img, beta, 0)

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{i:09d}.jpg")

    # save output image
    cv2.imwrite(out_img_path, out_img)

img: (10000, 24000), ref_img: (808, 1920)
img_ratio: 2.4, ref_img_ratio: 2.376237623762376


In [6]:
from core.utils.renderer.giffer import create_gif

In [8]:
create_gif(img_dir=project.out_img_dir_dict[out_img_dir], out_path=project.project_dir/"blend_mosaic_with_original.gif")

In [ ]:
# set reference shape
ref_img_height, ref_img_width, ref_img_channel = 50, 50, 3

# set output grid image shape
out_img_height = ref_img_height * NB_ROWS
out_img_width = ref_img_width * NB_COLS
out_img_channel = ref_img_channel

# read images to use to recreate dest image
src_imgs = [resize_with_crop(img=cv2.cvtColor(cv2.imread(src_img_path), cv2.COLOR_BGR2LAB), ref_img_shape=(ref_img_width, ref_img_height, ref_img_channel)) for src_img_path in project.in_img_path_list[:5]]

# get dest image and resize to output grid image shape
dest_img = resize_with_crop(img=cv2.cvtColor(cv2.imread(dest_img_path), cv2.COLOR_BGR2LAB), ref_img_shape=(out_img_width, out_img_height, out_img_channel))

dest_img = cv2.bilateralFilter(dest_img,9,75,75)